# 1000 AI-Generated Faces — 5 Unseen Generators

Generates **200 faces each** from 5 diffusion models NOT in the training set.
Total: **1000 images.** All from community mirrors (no HuggingFace login needed).

**Setup:** `Runtime → Change runtime type → T4 GPU`

| # | Folder | Model | Why it's 'unseen' |
|---|--------|-------|-------------------|
| 1 | `01_SD-1.5` | Stable Diffusion 1.5 | Older than SDXL |
| 2 | `02_SD-2.1` | Stable Diffusion 2.1 | Different text encoder |
| 3 | `03_RealisticVision-V6` | RealisticVision V6.0 | Community fine-tune |
| 4 | `04_Playground-v2.5` | Playground v2.5 | Modern aesthetic model |
| 5 | `05_SDXL-base-1.0` | SDXL base 1.0 | Different from training SDXL |

Runtime: ~2.5–3 hrs on T4. Split across 2 sessions if needed.

**No Kaggle upload.** At the end, the full dataset is zipped and downloaded to your machine.

In [ ]:
!pip install -q diffusers transformers accelerate safetensors scipy
print('Dependencies installed.')

Dependencies installed.


In [ ]:
import os, random, json, gc, time, csv, shutil
import torch
import numpy as np
from PIL import Image

BASE_DIR = '/content/ai_faces_dataset'
os.makedirs(BASE_DIR, exist_ok=True)

IMAGES_PER_MODEL = 200   # 5 models x 200 = 1000 faces
FACE_SIZE = 512

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', device)
if device == 'cpu':
    raise SystemExit('No GPU. Enable GPU runtime first.')

# Master manifest
MANIFEST = os.path.join(BASE_DIR, 'manifest.csv')
if not os.path.exists(MANIFEST):
    with open(MANIFEST, 'w', newline='') as f:
        csv.writer(f).writerow(['model', 'filename', 'prompt', 'seed'])

def make_folder(name):
    p = os.path.join(BASE_DIR, name)
    os.makedirs(p, exist_ok=True)
    return p

def log_entry(model_name, filename, prompt, seed):
    with open(MANIFEST, 'a', newline='') as f:
        csv.writer(f).writerow([model_name, filename, prompt, seed])

def free_memory(*objs):
    for o in objs:
        try: del o
        except Exception: pass
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def clear_hf_cache():
    cache = os.path.expanduser('~/.cache/huggingface/hub')
    if os.path.exists(cache):
        shutil.rmtree(cache, ignore_errors=True)
        print('  HF cache cleared.')

def is_valid_face(img):
    arr = np.asarray(img.convert('RGB'))
    if arr.std() < 10: return False
    if arr.mean() < 5 or arr.mean() > 250: return False
    return True

# Prompt builder
GENDERS     = ['man', 'woman']
AGES        = ['in their early 20s', 'in their 30s', 'in their 40s', 'in their 50s',
               'in their 60s', 'young adult', 'elderly']
ETHNICITIES = ['African', 'Asian', 'Caucasian', 'Hispanic', 'Middle Eastern',
               'South Asian', 'East Asian', 'mixed-race']
EXPRESSIONS = ['neutral expression', 'gentle smile', 'slight smile',
               'calm expression', 'confident expression']
LIGHTING    = ['soft studio lighting', 'natural window light',
               'professional headshot lighting', 'even softbox lighting']
QUALITY     = ['realistic skin texture', 'high detail', 'sharp focus',
               'photorealistic', 'DSLR photo, 85mm lens', '8k, ultra detailed']

def build_face_prompt():
    return (
        f'close-up portrait photograph of a {random.choice(AGES)} '
        f'{random.choice(ETHNICITIES)} {random.choice(GENDERS)}, '
        f'{random.choice(EXPRESSIONS)}, {random.choice(LIGHTING)}, '
        f'{random.choice(QUALITY)}, centered face, looking at camera, '
        f'plain neutral background'
    )

NEGATIVE_PROMPT = (
    'cartoon, illustration, painting, drawing, anime, 3d render, deformed, '
    'disfigured, extra limbs, bad anatomy, blurry, low quality, watermark, '
    'text, logo, multiple faces'
)

print('Setup complete. Base dir:', BASE_DIR)

Device: cuda
Setup complete. Base dir: /content/ai_faces_dataset


## Model 1 — Stable Diffusion 1.5
Using community mirror to avoid gating issues.

In [ ]:
from diffusers import StableDiffusionPipeline

folder = make_folder('01_SD-1.5')
pipe = StableDiffusionPipeline.from_pretrained(
    'stable-diffusion-v1-5/stable-diffusion-v1-5',
    torch_dtype=torch.float16,
    safety_checker=None,
    requires_safety_checker=False,
).to(device)
pipe.set_progress_bar_config(disable=True)
pipe.enable_attention_slicing()

t0 = time.time()
saved = 0
attempts = 0
while saved < IMAGES_PER_MODEL and attempts < IMAGES_PER_MODEL * 3:
    attempts += 1
    prompt = build_face_prompt()
    seed = random.randint(0, 2**32 - 1)
    gen = torch.Generator(device=device).manual_seed(seed)
    img = pipe(prompt, negative_prompt=NEGATIVE_PROMPT,
               height=FACE_SIZE, width=FACE_SIZE,
               num_inference_steps=30, guidance_scale=7.5,
               generator=gen).images[0]
    if not is_valid_face(img):
        continue
    fname = f'sd15_{saved+1:04d}.png'
    img.save(os.path.join(folder, fname))
    log_entry('SD-1.5', fname, prompt, seed)
    saved += 1
    if saved % 20 == 0:
        print(f'  SD1.5: {saved}/{IMAGES_PER_MODEL}  ({time.time()-t0:.0f}s)')

print(f'SD-1.5 done: {saved} images in {time.time()-t0:.1f}s')
free_memory(pipe)
clear_hf_cache()

/usr/local/lib/python3.13/dist-packages/diffusers/utils/deprecation_utils.py:23: FutureWarning: `torch_dtype` is deprecated and will be removed in version 1.0.0. Please use `dtype` instead.
  deprecate("torch_dtype", "1.0.0", _TORCH_DTYPE_DEPRECATION_MESSAGE)


model_index.json:   0%|          | 0.00/541 [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 13 files:   0%|          | 0/13 [00:00<?, ?it/s]

Loading pipeline components...:   0%|          | 0/6 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

  SD1.5: 20/200  (147s)
  SD1.5: 40/200  (304s)
  SD1.5: 60/200  (461s)
  SD1.5: 80/200  (619s)
  SD1.5: 100/200  (775s)
  SD1.5: 120/200  (933s)
  SD1.5: 140/200  (1091s)
  SD1.5: 160/200  (1248s)
  SD1.5: 180/200  (1406s)
  SD1.5: 200/200  (1564s)
SD-1.5 done: 200 images in 1564.1s
  HF cache cleared.


## Model 2 — Stable Diffusion 2.1 (community mirror)

In [ ]:
from diffusers import StableDiffusionPipeline

folder = make_folder('02_SD-2.1')
pipe = StableDiffusionPipeline.from_pretrained(
    'sd2-community/stable-diffusion-2-1-base',
    torch_dtype=torch.float16,
    safety_checker=None,
    requires_safety_checker=False,
).to(device)
pipe.set_progress_bar_config(disable=True)
pipe.enable_attention_slicing()

t0 = time.time()
saved = 0
attempts = 0
while saved < IMAGES_PER_MODEL and attempts < IMAGES_PER_MODEL * 3:
    attempts += 1
    prompt = build_face_prompt()
    seed = random.randint(0, 2**32 - 1)
    gen = torch.Generator(device=device).manual_seed(seed)
    img = pipe(prompt, negative_prompt=NEGATIVE_PROMPT,
               height=FACE_SIZE, width=FACE_SIZE,
               num_inference_steps=30, guidance_scale=7.5,
               generator=gen).images[0]
    if not is_valid_face(img):
        continue
    fname = f'sd21_{saved+1:04d}.png'
    img.save(os.path.join(folder, fname))
    log_entry('SD-2.1', fname, prompt, seed)
    saved += 1
    if saved % 20 == 0:
        print(f'  SD2.1: {saved}/{IMAGES_PER_MODEL}  ({time.time()-t0:.0f}s)')

print(f'SD-2.1 done: {saved} images in {time.time()-t0:.1f}s')
free_memory(pipe)
clear_hf_cache()

model_index.json:   0%|          | 0.00/543 [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 13 files:   0%|          | 0/13 [00:00<?, ?it/s]

Loading pipeline components...:   0%|          | 0/6 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/372 [00:00<?, ?it/s]

  SD2.1: 20/200  (139s)
  SD2.1: 40/200  (277s)
  SD2.1: 60/200  (414s)
  SD2.1: 80/200  (553s)
  SD2.1: 100/200  (691s)
  SD2.1: 120/200  (828s)
  SD2.1: 140/200  (966s)
  SD2.1: 160/200  (1102s)
  SD2.1: 180/200  (1240s)
  SD2.1: 200/200  (1377s)
SD-2.1 done: 200 images in 1377.2s
  HF cache cleared.


## Model 3 — RealisticVision V6 (community mirror)

In [ ]:
from diffusers import DiffusionPipeline

folder = make_folder('03_RealisticVision-V6')
pipe = DiffusionPipeline.from_pretrained(
    'SirSatorik/RealisticVisionV6.0B1',
    torch_dtype=torch.float16,
    safety_checker=None,
    requires_safety_checker=False,
).to(device)
pipe.set_progress_bar_config(disable=True)
pipe.enable_attention_slicing()

t0 = time.time()
saved = 0
attempts = 0
while saved < IMAGES_PER_MODEL and attempts < IMAGES_PER_MODEL * 3:
    attempts += 1
    prompt = build_face_prompt() + ', RAW photo, film grain, Fujifilm XT3'
    seed = random.randint(0, 2**32 - 1)
    gen = torch.Generator(device=device).manual_seed(seed)
    img = pipe(prompt, negative_prompt=NEGATIVE_PROMPT,
               height=FACE_SIZE, width=FACE_SIZE,
               num_inference_steps=25, guidance_scale=6.0,
               generator=gen).images[0]
    if not is_valid_face(img):
        continue
    fname = f'realvis_{saved+1:04d}.png'
    img.save(os.path.join(folder, fname))
    log_entry('RealisticVision-V6', fname, prompt, seed)
    saved += 1
    if saved % 20 == 0:
        print(f'  RealisticVision: {saved}/{IMAGES_PER_MODEL}  ({time.time()-t0:.0f}s)')

print(f'RealisticVision done: {saved} images in {time.time()-t0:.1f}s')
free_memory(pipe)
clear_hf_cache()

model_index.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 13 files:   0%|          | 0/13 [00:00<?, ?it/s]

Loading pipeline components...:   0%|          | 0/6 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

CLIPFeatureExtractor appears to have been deprecated in transformers. Using CLIPImageProcessor instead.


  RealisticVision: 20/200  (134s)
  RealisticVision: 40/200  (267s)
  RealisticVision: 60/200  (400s)
  RealisticVision: 80/200  (533s)
  RealisticVision: 100/200  (666s)
  RealisticVision: 120/200  (803s)
  RealisticVision: 140/200  (936s)
  RealisticVision: 160/200  (1068s)
  RealisticVision: 180/200  (1202s)
  RealisticVision: 200/200  (1335s)
RealisticVision done: 200 images in 1334.8s
  HF cache cleared.


## Model 4 — Playground v2.5

In [ ]:
from diffusers import DiffusionPipeline

folder = make_folder('04_Playground-v2.5')
pipe = DiffusionPipeline.from_pretrained(
    'playgroundai/playground-v2.5-1024px-aesthetic',
    torch_dtype=torch.float16,
    variant='fp16',
).to(device)
pipe.set_progress_bar_config(disable=True)
pipe.enable_attention_slicing()

t0 = time.time()
saved = 0
attempts = 0
while saved < IMAGES_PER_MODEL and attempts < IMAGES_PER_MODEL * 3:
    attempts += 1
    prompt = build_face_prompt()
    seed = random.randint(0, 2**32 - 1)
    gen = torch.Generator(device=device).manual_seed(seed)
    img = pipe(prompt, negative_prompt=NEGATIVE_PROMPT,
               height=FACE_SIZE, width=FACE_SIZE,
               num_inference_steps=30, guidance_scale=3.0,
               generator=gen).images[0]
    if not is_valid_face(img):
        continue
    fname = f'playground_{saved+1:04d}.png'
    img.save(os.path.join(folder, fname))
    log_entry('Playground-v2.5', fname, prompt, seed)
    saved += 1
    if saved % 20 == 0:
        print(f'  Playground: {saved}/{IMAGES_PER_MODEL}  ({time.time()-t0:.0f}s)')

print(f'Playground done: {saved} images in {time.time()-t0:.1f}s')
free_memory(pipe)
clear_hf_cache()

model_index.json:   0%|          | 0.00/684 [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 18 files:   0%|          | 0/18 [00:00<?, ?it/s]

Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/517 [00:00<?, ?it/s]

...

There are modules in AutoencoderKL that should be kept in float32: []. Casting directly with `to()` can lead to inconsistent results; set `torch_dtype` in `from_pretrained()` instead to keep these modules in float32.
There are modules in AutoencoderKL that should be kept in float32: []. Casting directly with `to()` can lead to inconsistent results; set `torch_dtype` in `from_pretrained()` instead to keep these modules in float32.
There are modules in AutoencoderKL that should be kept in float32: []. Casting directly with `to()` can lead to inconsistent results; set `torch_dtype` in `from_pretrained()` instead to keep these modules in float32.
There are modules in AutoencoderKL that should be kept in float32: []. Casting directly with `to()` can lead to inconsistent results; set `torch_dtype` in `from_pretrained()` instead to keep these modules in float32.
There are modules in AutoencoderKL that should be kept in float32: []. Casting directly with `to()` can lead to inconsistent results

  Playground: 120/200  (1125s)


There are modules in AutoencoderKL that should be kept in float32: []. Casting directly with `to()` can lead to inconsistent results; set `torch_dtype` in `from_pretrained()` instead to keep these modules in float32.
There are modules in AutoencoderKL that should be kept in float32: []. Casting directly with `to()` can lead to inconsistent results; set `torch_dtype` in `from_pretrained()` instead to keep these modules in float32.
There are modules in AutoencoderKL that should be kept in float32: []. Casting directly with `to()` can lead to inconsistent results; set `torch_dtype` in `from_pretrained()` instead to keep these modules in float32.
There are modules in AutoencoderKL that should be kept in float32: []. Casting directly with `to()` can lead to inconsistent results; set `torch_dtype` in `from_pretrained()` instead to keep these modules in float32.
There are modules in AutoencoderKL that should be kept in float32: []. Casting directly with `to()` can lead to inconsistent results

  Playground: 140/200  (1312s)


There are modules in AutoencoderKL that should be kept in float32: []. Casting directly with `to()` can lead to inconsistent results; set `torch_dtype` in `from_pretrained()` instead to keep these modules in float32.
There are modules in AutoencoderKL that should be kept in float32: []. Casting directly with `to()` can lead to inconsistent results; set `torch_dtype` in `from_pretrained()` instead to keep these modules in float32.
There are modules in AutoencoderKL that should be kept in float32: []. Casting directly with `to()` can lead to inconsistent results; set `torch_dtype` in `from_pretrained()` instead to keep these modules in float32.
There are modules in AutoencoderKL that should be kept in float32: []. Casting directly with `to()` can lead to inconsistent results; set `torch_dtype` in `from_pretrained()` instead to keep these modules in float32.
There are modules in AutoencoderKL that should be kept in float32: []. Casting directly with `to()` can lead to inconsistent results

## Model 5 — SDXL base 1.0

In [ ]:
from diffusers import AutoPipelineForText2Image

folder = make_folder('05_SDXL-base-1.0')
pipe = AutoPipelineForText2Image.from_pretrained(
    'stabilityai/stable-diffusion-xl-base-1.0',
    torch_dtype=torch.float16,
    variant='fp16',
).to(device)
pipe.set_progress_bar_config(disable=True)
pipe.enable_attention_slicing()

t0 = time.time()
saved = 0
attempts = 0
while saved < IMAGES_PER_MODEL and attempts < IMAGES_PER_MODEL * 3:
    attempts += 1
    prompt = build_face_prompt()
    seed = random.randint(0, 2**32 - 1)
    gen = torch.Generator(device=device).manual_seed(seed)
    img = pipe(prompt, negative_prompt=NEGATIVE_PROMPT,
               height=FACE_SIZE, width=FACE_SIZE,
               num_inference_steps=30, guidance_scale=7.5,
               generator=gen).images[0]
    if not is_valid_face(img):
        continue
    fname = f'sdxl_{saved+1:04d}.png'
    img.save(os.path.join(folder, fname))
    log_entry('SDXL-base-1.0', fname, prompt, seed)
    saved += 1
    if saved % 20 == 0:
        print(f'  SDXL: {saved}/{IMAGES_PER_MODEL}  ({time.time()-t0:.0f}s)')

print(f'SDXL done: {saved} images in {time.time()-t0:.1f}s')
free_memory(pipe)
clear_hf_cache()

## Verify dataset

In [ ]:
total = 0
for folder in sorted(os.listdir(BASE_DIR)):
    fpath = os.path.join(BASE_DIR, folder)
    if os.path.isdir(fpath):
        n = len([f for f in os.listdir(fpath) if f.lower().endswith('.png')])
        total += n
        print(f'{folder:30s} -> {n} images')
print('-' * 45)
print(f'TOTAL: {total} images (expected {IMAGES_PER_MODEL * 5})')
print(f'Manifest: {MANIFEST}')

## (Optional) Download Real Faces from LFW
Adds 1000 real faces from a new source (not FFHQ) so your external test set has both classes.

In [ ]:
from sklearn.datasets import fetch_lfw_people
from PIL import Image
import numpy as np, os

folder = make_folder('00_REAL_LFW')
lfw = fetch_lfw_people(min_faces_per_person=1, resize=1.0, color=True)
print('LFW loaded:', lfw.images.shape)

n = min(1000, len(lfw.images))
for i in range(n):
    arr = (lfw.images[i] * 255).astype('uint8')
    Image.fromarray(arr).resize((FACE_SIZE, FACE_SIZE)).save(
        os.path.join(folder, f'real_lfw_{i+1:04d}.png')
    )
print(f'Saved {n} real faces from LFW to {folder}')

## Zip & Download the Full Dataset

Creates `ai_faces_dataset.zip` and downloads it to your local machine.

In [ ]:
import shutil, os
from google.colab import files

ZIP_PATH = '/content/ai_faces_dataset'
print('Zipping dataset (this may take 1-2 min)...')
shutil.make_archive(ZIP_PATH, 'zip', BASE_DIR)
zip_file = ZIP_PATH + '.zip'
size_mb = os.path.getsize(zip_file) / (1024 * 1024)
print(f'Zip created: {zip_file}  ({size_mb:.1f} MB)')

print('Starting download...')
files.download(zip_file)